In [1]:
import urllib.request

# GDELT daily GKG bulk file format
# Example: https://data.gdeltproject.org/gkg/20200101.gkg.csv.zip
test_url = "http://data.gdeltproject.org/gkg/20200101.gkg.csv.zip"
try:
    with urllib.request.urlopen(test_url, timeout=10) as r:
        print("Status:", r.status)
        print("Content-Length:", r.headers.get("Content-Length"))
except Exception as e:
    print("Error:", e)

Status: 200
Content-Length: 20015562


In [3]:
import pandas as pd
from pathlib import Path
import urllib.request

test_day = "20200101"
url = f"http://data.gdeltproject.org/gkg/{test_day}.gkg.csv.zip"

# Use a real Windows-friendly temp path
dest = Path.home() / "Downloads" / f"{test_day}.gkg.csv.zip"

urllib.request.urlretrieve(url, dest)
print("Downloaded:", dest.stat().st_size / 1e6, "MB")

# Read just first 1000 rows to check structure
df = pd.read_csv(dest, sep="\t", header=None, nrows=1000,
                 usecols=[0,1,3,4,7,26],
                 names=["gkg_record_id","date","source_common_name",
                        "document_identifier","v1_themes","extras_xml"],
                 dtype="string", on_bad_lines="skip")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nSample theme:", df["v1_themes"].dropna().iloc[0])

Downloaded: 20.015562 MB


ParserError: Defining usecols with out-of-bounds indices is not allowed. [26] are out of bounds.

In [4]:
# Step 1: count how many columns the daily file actually has
df_raw = pd.read_csv(dest, sep="\t", header=None, nrows=5,
                     dtype="string", on_bad_lines="skip")
print("Number of columns:", df_raw.shape[1])
print(df_raw.iloc[0])

Number of columns: 11
0              DATE
1           NUMARTS
2            COUNTS
3            THEMES
4         LOCATIONS
5           PERSONS
6     ORGANIZATIONS
7              TONE
8     CAMEOEVENTIDS
9           SOURCES
10       SOURCEURLS
Name: 0, dtype: string


In [5]:
# Test GKG 2.0 daily bulk URL
test_url_v2 = "http://data.gdeltproject.org/gdeltv2/20200101.gkg.csv.zip"
try:
    with urllib.request.urlopen(test_url_v2, timeout=10) as r:
        print("Status:", r.status)
        print("Content-Length:", int(r.headers.get("Content-Length", 0)) / 1e6, "MB")
except Exception as e:
    print("Error:", e)

Error: HTTP Error 404: Not Found


In [6]:
df_raw = pd.read_csv(dest, sep="\t", header=None, nrows=20,
                     names=["DATE","NUMARTS","COUNTS","THEMES","LOCATIONS",
                             "PERSONS","ORGANIZATIONS","TONE",
                             "CAMEOEVENTIDS","SOURCES","SOURCEURLS"],
                     dtype="string", on_bad_lines="skip")

# Check themes - should look like WB_1096_COAL;ENV_CLIMATECHANGE; etc
print("=== THEMES sample ===")
print(df_raw["THEMES"].dropna().head(5).to_string())

print("\n=== SOURCEURLS sample ===")
print(df_raw["SOURCEURLS"].dropna().head(3).to_string())

print("\n=== DATE sample ===")
print(df_raw["DATE"].head(3).to_string())

print("\n=== TONE sample ===")
print(df_raw["TONE"].head(3).to_string())

=== THEMES sample ===
0                                               THEMES
2    MILITARY;TAX_WORLDLANGUAGES;TAX_WORLDLANGUAGES...
3    TAX_ECON_PRICE;FUELPRICES;ECON_GASOLINEPRICE;E...
4    GENERAL_HEALTH;MEDICAL;SOC_POINTSOFINTEREST;SO...
5    MEDIA_MSM;WB_678_DIGITAL_GOVERNMENT;WB_694_BRO...

=== SOURCEURLS sample ===
0                                           SOURCEURLS
1    https://www.thisiswiltshire.co.uk/news/headlin...
2    https://www.deccanchronicle.com/nation/current...

=== DATE sample ===
0        DATE
1    20200101
2    20200101

=== TONE sample ===
0                                                 TONE
1    -1.86480186480187,1.63170163170163,3.496503496...
2    -8.58895705521472,0,8.58895705521472,8.5889570...


In [13]:
import subprocess
subprocess.run(["pip", "install", "pyarrow", "--quiet"], check=True)
print("Done")

Done


In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.insert(0, str(project_root / "src"))

import importlib
import gdelt_article_audit
importlib.reload(gdelt_article_audit)

from gdelt_article_audit import build_gdelt_news_dataset
from paths import RAW, PROCESSED

panel = build_gdelt_news_dataset(
    start_day="2020-01-02",
    end_day="2020-01-06",
    raw_news_dir=RAW / "news",
    processed_dir=PROCESSED / "news",
    download_first=False,   # ya están descargados
    save_article_level=True
)

print(panel[["day","n_articles_raw","n_articles_dedup",
             "n_relevant_articles","share_relevant_articles",
             "energy_supply_geopolitics_count","tone_mean"]].to_string())


=== 2020-01-02 ===
raw=75,610 | dedup=74,222 | relevant=65,904

=== 2020-01-03 ===
raw=83,566 | dedup=82,069 | relevant=73,516

=== 2020-01-04 ===
raw=50,315 | dedup=49,311 | relevant=43,759

=== 2020-01-05 ===
raw=46,103 | dedup=45,040 | relevant=39,789

=== 2020-01-06 ===
raw=87,238 | dedup=85,702 | relevant=77,129
          day  n_articles_raw  n_articles_dedup  n_relevant_articles  share_relevant_articles  energy_supply_geopolitics_count tone_mean
0  2020-01-02           75610             74222                65904                 0.887931                            25493      None
1  2020-01-03           83566             82069                73516                 0.895783                            32365      None
2  2020-01-04           50315             49311                43759                 0.887408                            20117      None
3  2020-01-05           46103             45040                39789                 0.883415                            18870      

In [ ]:
importlib.reload(gdelt_article_audit)
from gdelt_article_audit import read_gkg_zip, clean_gkg_frame, add_relevance_flags
from paths import RAW

zip_path = RAW / "news" / "20200102.gkg.csv.zip"
df = read_gkg_zip(zip_path)
df = clean_gkg_frame(df)
df = add_relevance_flags(df)

channel_cols = ["transition_policy","physical_risk",
                "energy_supply_geopolitics","clean_tech","activism_litigation"]
print("=== Conteo por canal ===")
print(df[channel_cols].sum().sort_values(ascending=False))
print(f"\nTotal relevantes: {df['candidate_relevant'].sum():,} de {len(df):,}")
print(f"Share: {df['candidate_relevant'].mean():.1%}")

print("\n=== Sanity check: 5 artículos relevantes ===")
sample = df[df["candidate_relevant"]][["v1_themes","energy_supply_geopolitics",
                                        "transition_policy","physical_risk"]].head(5)
for _, row in sample.iterrows():
    flags = [c for c in channel_cols if row.get(c, False)]
    print(f"  [{', '.join(flags)}]")
    themes_short = ";".join(row["v1_themes"].split(";")[:6])
    print(f"  {themes_short}\n")

=== Conteo por canal ===
physical_risk                6438
energy_supply_geopolitics    5154
transition_policy            2617
clean_tech                   1099
activism_litigation             0
dtype: Int64

Total relevantes: 12,176 de 75,610
Share: 16.1%

=== Sanity check: 5 artículos relevantes ===
  [physical_risk]
  MANMADE_DISASTER_IMPLIED;WB_168_ROADS_AND_HIGHWAYS;WB_135_TRANSPORT;WB_1809_HIGHWAYS;WB_1803_TRANSPORT_INFRASTRUCTURE;MEDIA_SOCIAL

  [energy_supply_geopolitics]
  TAX_FNCACT;TAX_FNCACT_CHEFS;TAX_FNCACT_CHEF;TAX_ETHNICITY;TAX_ETHNICITY_AMERICAN;WB_2931_IRON

  [transition_policy, physical_risk]
  MANMADE_DISASTER_IMPLIED;UNGP_FORESTS_RIVERS_OCEANS;WB_698_TRADE;SANCTIONS;CRISISLEX_CRISISLEXREC;GENERAL_HEALTH

  [physical_risk]
  NATURAL_DISASTER;NATURAL_DISASTER_SEVERE_WEATHER;NATURAL_DISASTER_TORNADOES;CRISISLEX_O01_WEATHER;CRISISLEX_CRISISLEXREC;TAX_FNCACT

  [energy_supply_geopolitics]
  TAX_FNCACT;TAX_FNCACT_MAN;TAX_FNCACT_BUSINESSMAN;TAX_FNCACT_INDUSTRIALIST;TAX_RE

In [ ]:
importlib.reload(gdelt_article_audit)
from gdelt_article_audit import read_gkg_zip, clean_gkg_frame, add_relevance_flags
from paths import RAW

df = read_gkg_zip(RAW / "news" / "20200102.gkg.csv.zip")
df = clean_gkg_frame(df)
df = add_relevance_flags(df)

channel_cols = ["transition_policy","physical_risk",
                "energy_supply_geopolitics","clean_tech","activism_litigation"]
print(df[channel_cols].sum().sort_values(ascending=False))
print(f"\nShare relevante: {df['candidate_relevant'].mean():.1%}")

physical_risk                3337
transition_policy            2617
energy_supply_geopolitics    1707
clean_tech                   1099
activism_litigation             0
dtype: Int64

Share relevante: 9.6%
